# 1.1 — Introduction to Generative AI

**Module 01 · Intro to GenAI**

In this notebook you'll make your first call to a **Large Language Model (LLM)** running locally with [Ollama](https://ollama.com), and build a tiny customer-support assistant.

### Learning objectives
By the end of this notebook you will be able to:
1. Send a prompt to a local LLM from Python.
2. Explain the difference between a **system** message and a **user** message.
3. **Stream** a model's response token-by-token.
4. Change the model's behaviour just by editing the system prompt.

### Prerequisites
| Requirement | How to check |
|---|---|
| Ollama installed and running | `ollama --version` |
| A model pulled locally | `ollama pull gemma3:1b` |
| Python package | `uv add ollama` |

---
## 1. Setup

Import the `chat` function from the Ollama Python client. It talks to the Ollama server on `http://localhost:11434` by default.

In [1]:
from ollama import chat

## 2. Configuration

We keep the settings in one place so they're easy to change:

- **`MODEL`** — which LLM to use. `gemma3:1b` is small and fast enough for a laptop.
- **`SYSTEM_PROMPT`** — instructions that shape *how* the model behaves for every request.

> **Tip:** Try swapping `MODEL` for a larger model (e.g. `"gemma4:31b-cloud"`) and compare the answers.

In [2]:
MODEL = "gemma3:1b"
# MODEL = "gemma4:31b-cloud"

SYSTEM_PROMPT = (
    "You are a professional customer support assistant for an online store. "
    "Reply politely, give clear next steps, and keep answers under 80 words."
)

## 3. Messages: system vs. user

Chat models take a **list of messages**, each with a `role`:

| Role | Purpose | Example |
|---|---|---|
| `system` | Sets the assistant's persona, rules and tone | *"You are a support assistant…"* |
| `user` | The actual question or request | *"My order arrived damaged."* |
| `assistant` | The model's previous replies (used for multi-turn chat) | — |

## 4. Build the support assistant

`ask_support()` wraps the model call. With `stream=True`, the response arrives in small **chunks** that we print as they come — that's what creates the "typing" effect you see in ChatGPT and Claude.

In [3]:
def ask_support(question: str) -> str:
    """Ask the support assistant a question, streaming the reply as it is generated."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    stream = chat(model=MODEL, messages=messages, stream=True)

    reply = ""
    for chunk in stream:
        piece = chunk.message.content
        print(piece, end="", flush=True)
        reply += piece
    print()
    return reply

## 5. Try it out

Run the cell below and watch the answer stream in.

In [4]:
answer = ask_support("My order arrived damaged. What should I do?")

We're so sorry your order arrived damaged! 

First, please reply to this email with a picture of the damage - that'll speed up the process. 

In the meantime, we’ll offer you a full refund, or a replacement free of charge.  Would you prefer a refund or replacement? Thanks 😊


Because the function also **returns** the full reply, we can inspect it afterwards — for example, to check the model respected the 80-word limit.

In [5]:
print(f"Word count: {len(answer.split())}")

Word count: 49


## 6. More questions

Let's see how the same system prompt handles different requests.

In [7]:
questions = [
    "How long does shipping usually take?",
    "Can I change the delivery address after placing an order?",
    "I was charged twice for the same order.",
]

for q in questions:
    print(f"U {q}")
    print("A ", end="")
    ask_support(q)
    print("-" * 60)

U How long does shipping usually take?
A Hello! Thank you for your question! Shipping times depend on several factors, but typically it takes **3-8 business days** after we confirm your order. Here’s a general timeline:

**Here's what you’ll need:**

*   **Shipping Destination:**  Where are you located? 
*   **Order Weight and Size:** Knowing this allows us to estimate delivery time. 

Let’s tackle this! 😊
------------------------------------------------------------
U Can I change the delivery address after placing an order?
A Hi there! Absolutely yes! At any point after processing your order, you can update your address.

**Follow these steps:**

1. Go to your "Order History" or "My Account" page.
2. Find the order containing the change.
3. Click on “Edit Information” and enter your new address. 


Do you need help with a specific step?
------------------------------------------------------------
U I was charged twice for the same order.
A Hi there! I’m sincerely sorry to hear this. L

---
## Exercises

1. **Change the persona.** Edit `SYSTEM_PROMPT` so the assistant replies like a friendly pirate. Re-run the cells — what changes, and what stays the same?
2. **Tighten the rules.** Limit replies to **30 words** and require a bullet-point list of next steps. Does the model obey?
3. **Off-topic test.** Ask `ask_support("What's the capital of France?")`. Add a rule to the system prompt so the assistant politely refuses non-store questions.
4. **Compare models.** Switch `MODEL` to a larger model and compare quality and speed.

In [ ]:
# Your experiments here

---
## Summary

- An LLM is called with a **list of messages**, each with a `role`.
- The **system prompt** controls tone, persona and rules — without changing any code.
- **Streaming** prints the response chunk-by-chunk for a responsive experience.
- Running models locally with **Ollama** means no API keys and no cost.

**Next:** [2.1 — Tokens and Tokenization](../02-how-llms-work/2.1-tokens-and-tokenization.ipynb) — how LLMs actually read text.